## Nonlinear MLP Probe Baseline

This supervised diagnostic asks whether hidden residual information is present in the full residual vector in a nonlinear form. It trains one multi-output MLP on `train/res_mu`, selects the checkpoint by validation mean hidden-concept AUC, and reports held-out test AUC per hidden residual.

Interpretation: if `mlp_auc` is much higher than `distributed_auc`, the residual channel contains nonlinear hidden information. If it is similar, the linear dense probe is already a good upper-bound diagnostic.


In [ ]:
# -----------------------------
# Nonlinear MLP probe recovery
# -----------------------------
# This is a supervised upper-bound diagnostic, not a discovery method. It uses
# hidden labels to ask whether each hidden residual is nonlinearly decodable from
# the full residual channel.

MLP_PROBE_CONFIG = {
    "hidden_dims": [64, 64],
    "dropout": 0.1,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "batch_size": 1024,
    "max_epochs": 200,
    "patience": 20,
    "seed": 0,
}


class ResidualMLPProbe(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_dims=(64, 64), dropout=0.1):
        super().__init__()
        layers = []
        prev = in_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev = hidden_dim
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def standardize_numpy_from_train(train_scores, val_scores, test_scores):
    X_train = to_numpy(train_scores).astype(np.float32)
    X_val = to_numpy(val_scores).astype(np.float32)
    X_test = to_numpy(test_scores).astype(np.float32)
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-8
    return (X_train - mean) / std, (X_val - mean) / std, (X_test - mean) / std


def auc_per_hidden(y_true, y_score):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    aucs = []
    for h in range(y_true.shape[1]):
        if len(np.unique(y_true[:, h])) < 2:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(y_true[:, h], y_score[:, h]))
    return np.asarray(aucs, dtype=float)


def train_mlp_probe_train_val_test(
    train_scores,
    train_hidden,
    val_scores,
    val_hidden,
    test_scores,
    test_hidden,
    config=MLP_PROBE_CONFIG,
):
    torch.manual_seed(config["seed"])
    np.random.seed(config["seed"])

    X_train, X_val, X_test = standardize_numpy_from_train(train_scores, val_scores, test_scores)
    H_train = to_numpy(train_hidden).astype(np.float32)
    H_val = to_numpy(val_hidden).astype(np.float32)
    H_test = to_numpy(test_hidden).astype(np.float32)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = ResidualMLPProbe(
        in_dim=X_train.shape[1],
        out_dim=H_train.shape[1],
        hidden_dims=config["hidden_dims"],
        dropout=config["dropout"],
    ).to(device)

    n_pos = H_train.sum(axis=0)
    n_neg = H_train.shape[0] - n_pos
    pos_weight = torch.tensor(n_neg / np.maximum(n_pos, 1.0), dtype=torch.float32, device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])

    train_ds = torch.utils.data.TensorDataset(torch.from_numpy(X_train), torch.from_numpy(H_train))
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=config["batch_size"],
        shuffle=True,
    )

    X_val_t = torch.from_numpy(X_val).to(device)
    X_test_t = torch.from_numpy(X_test).to(device)

    best_state = None
    best_epoch = 0
    best_val_mean_auc = -np.inf
    best_val_auc_per_hidden = None
    stale_epochs = 0
    history = []

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        total_loss = 0.0
        total_n = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.item()) * xb.shape[0]
            total_n += xb.shape[0]

        model.eval()
        with torch.no_grad():
            val_prob = torch.sigmoid(model(X_val_t)).cpu().numpy()
        val_auc = auc_per_hidden(H_val, val_prob)
        # Average mean of AUC across hidden residuals
        val_mean_auc = float(np.nanmean(val_auc))
        history.append({"epoch": epoch, "train_loss": total_loss / total_n, "mean_val_auc": val_mean_auc})

        # Save model if validation mean AUC improves
        if val_mean_auc > best_val_mean_auc + 1e-5:
            best_val_mean_auc = val_mean_auc
            best_epoch = epoch
            best_val_auc_per_hidden = val_auc.copy()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1

        if epoch == 1 or epoch % 10 == 0:
            print(f"epoch {epoch:03d} | loss {total_loss / total_n:.5f} | mean val AUC {val_mean_auc:.4f}")

        if stale_epochs >= config["patience"]:
            break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_prob = torch.sigmoid(model(X_test_t)).cpu().numpy()
    # Calculate AUC based on true hidden concept labels and MLP probe predicted probabilities on the test set
    test_auc = auc_per_hidden(H_test, test_prob)
    test_pred = (test_prob >= 0.5).astype(int)

    rows = []
    for h in range(H_test.shape[1]):
        rows.append({
            "hidden_idx": h,
            "mlp_auc": float(test_auc[h]),
            "mlp_val_auc_at_selected_epoch": float(best_val_auc_per_hidden[h]),
            "mlp_accuracy": accuracy_score(H_test[:, h].astype(int), test_pred[:, h]),
            "mlp_f1": f1_score(H_test[:, h].astype(int), test_pred[:, h], zero_division=0),
        })

    metadata = {
        "best_epoch": best_epoch,
        "best_mean_val_auc": best_val_mean_auc,
        "epochs_run": len(history),
        "device": str(device),
    }
    return pd.DataFrame(rows), pd.DataFrame(history), metadata


mlp_probe_eval, mlp_probe_history, mlp_probe_metadata = train_mlp_probe_train_val_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["val"]["res_mu"],
    splits["val"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)

mlp_probe_recovery_eval = add_task_relevance(mlp_probe_eval, splits["test"]["w_hid"]).sort_values("rank_abs_w")

print("MLP probe metadata:", mlp_probe_metadata)
print("MLP probe recovery ranked by abs(w_hid)")
display(mlp_probe_recovery_eval[[
    "hidden_idx",
    "mlp_auc",
    "mlp_val_auc_at_selected_epoch",
    "mlp_accuracy",
    "mlp_f1",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
]])

print("MLP probe recovery among top-k hidden concepts by abs(w_hid)")
display(summarize_recovery_by_relevance(mlp_probe_recovery_eval, "mlp_auc", "abs_w_hid"))

print("Raw axis vs dense linear probe vs MLP probe")
mlp_probe_ladder = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(mlp_probe_recovery_eval[["hidden_idx", "mlp_auc"]], on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)
display(mlp_probe_ladder)

mlp_probe_ladder_summary = pd.DataFrame({
    "method": ["raw_axis", "distributed_probe", "mlp_probe"],
    "top1_mean_auc": [
        mlp_probe_ladder.head(1)["axis_auc"].mean(),
        mlp_probe_ladder.head(1)["distributed_auc"].mean(),
        mlp_probe_ladder.head(1)["mlp_auc"].mean(),
    ],
    "top3_mean_auc": [
        mlp_probe_ladder.head(3)["axis_auc"].mean(),
        mlp_probe_ladder.head(3)["distributed_auc"].mean(),
        mlp_probe_ladder.head(3)["mlp_auc"].mean(),
    ],
    "top5_mean_auc": [
        mlp_probe_ladder.head(5)["axis_auc"].mean(),
        mlp_probe_ladder.head(5)["distributed_auc"].mean(),
        mlp_probe_ladder.head(5)["mlp_auc"].mean(),
    ],
})
display(mlp_probe_ladder_summary)



epoch 001 | loss 0.68454 | mean val AUC 0.5811
epoch 010 | loss 0.66543 | mean val AUC 0.6218
epoch 020 | loss 0.65988 | mean val AUC 0.6334
epoch 030 | loss 0.65762 | mean val AUC 0.6397
epoch 040 | loss 0.65591 | mean val AUC 0.6428
epoch 050 | loss 0.65460 | mean val AUC 0.6456
epoch 060 | loss 0.65395 | mean val AUC 0.6469
epoch 070 | loss 0.65320 | mean val AUC 0.6483
epoch 080 | loss 0.65232 | mean val AUC 0.6495
epoch 090 | loss 0.65156 | mean val AUC 0.6506
epoch 100 | loss 0.65116 | mean val AUC 0.6512
epoch 110 | loss 0.65067 | mean val AUC 0.6519
epoch 120 | loss 0.65024 | mean val AUC 0.6524
epoch 130 | loss 0.64987 | mean val AUC 0.6525
epoch 140 | loss 0.64954 | mean val AUC 0.6536
epoch 150 | loss 0.64925 | mean val AUC 0.6536
epoch 160 | loss 0.64849 | mean val AUC 0.6536
epoch 170 | loss 0.64851 | mean val AUC 0.6540
epoch 180 | loss 0.64853 | mean val AUC 0.6547
epoch 190 | loss 0.64788 | mean val AUC 0.6547
epoch 200 | loss 0.64749 | mean val AUC 0.6548
MLP probe met

,hidden_idx,mlp_auc,mlp_val_auc_at_selected_epoch,mlp_accuracy,mlp_f1,w_hid,abs_w_hid,rank_abs_w
18,18,0.744336,0.737627,0.6816,0.669092,-0.705692,0.705692,1
2,2,0.763514,0.770617,0.6956,0.671345,-0.648612,0.648612,2
9,9,0.701821,0.684999,0.6457,0.631743,0.230210,0.230210,3
19,19,0.609039,0.614085,0.5738,0.538944,-0.150507,0.150507,4
7,7,0.655242,0.658235,0.6135,0.612142,0.075182,0.075182,5
4,4,0.685405,0.671147,0.6355,0.638859,0.000000,0.000000,6
5,5,0.671957,0.663654,0.6266,0.633202,0.000000,0.000000,6
6,6,0.685547,0.679996,0.6354,0.627503,0.000000,0.000000,6
8,8,0.655714,0.662954,0.6099,0.616760,0.000000,0.000000,6
1,1,0.615676,0.609506,0.5861,0.582383,0.000000,0.000000,6


MLP probe recovery among top-k hidden concepts by abs(w_hid)


,top_k_by,k,mean_mlp_auc,max_mlp_auc,num_mlp_auc_ge_0_7,hidden_indices
0,abs_w_hid,1,0.744336,0.744336,1,[18]
1,abs_w_hid,3,0.736557,0.763514,3,"[18, 2, 9]"
2,abs_w_hid,5,0.694790,0.763514,3,"[18, 2, 9, 19, 7]"
3,abs_w_hid,10,0.664185,0.763514,3,"[18, 2, 9, 19, 7, 12, 3, 17, 16, 15]"


Raw axis vs dense linear probe vs MLP probe


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,mlp_auc
0,18,0.700518,0.705692,1,0.728339,0.744336
1,2,0.731922,0.648612,2,0.754636,0.763514
2,9,0.671554,0.230210,3,0.695460,0.701821
3,19,0.544257,0.150507,4,0.586331,0.609039
4,7,0.601763,0.075182,5,0.639585,0.655242
17,17,0.574521,0.000000,6,0.618733,0.631111
16,16,0.576233,0.000000,6,0.610046,0.623104
15,15,0.522905,0.000000,6,0.641363,0.670067
14,14,0.533031,0.000000,6,0.596366,0.619535
13,13,0.627357,0.000000,6,0.668619,0.688388


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701332,0.650003
1,distributed_probe,0.728339,0.726145,0.680870
2,mlp_probe,0.744336,0.736557,0.694790


In [ ]:
task_relevant_hidden = splits["test"]["w_hid"].abs() > 1e-8

task_relevant_hidden_indices = list(torch.where(task_relevant_hidden)[0].cpu().numpy())
print("Task-relevant hidden residual indices:", task_relevant_hidden_indices)

print(splits["train"]["res_mu"].shape, splits["train"]["hidden_residuals"].shape)



mlp_probe_eval, mlp_probe_history, mlp_probe_metadata = train_mlp_probe_train_val_test(
    splits["train"]["res_mu"][:,task_relevant_hidden_indices],
    splits["train"]["hidden_residuals"][:,task_relevant_hidden_indices],
    splits["val"]["res_mu"][:,task_relevant_hidden_indices],
    splits["val"]["hidden_residuals"][:,task_relevant_hidden_indices],
    splits["test"]["res_mu"][:,task_relevant_hidden_indices],
    splits["test"]["hidden_residuals"][:,task_relevant_hidden_indices],
)


mlp_probe_recovery_eval = add_task_relevance(mlp_probe_eval, splits["test"]["w_hid"]).sort_values("rank_abs_w")

Task-relevant hidden residual indices: [np.int64(2), np.int64(7), np.int64(9), np.int64(18), np.int64(19)]
torch.Size([30000, 20]) torch.Size([30000, 20])
epoch 001 | loss 0.66580 | mean val AUC 0.6430
epoch 010 | loss 0.64635 | mean val AUC 0.6493
epoch 020 | loss 0.64433 | mean val AUC 0.6503
epoch 030 | loss 0.64395 | mean val AUC 0.6501
epoch 040 | loss 0.64344 | mean val AUC 0.6505
epoch 050 | loss 0.64324 | mean val AUC 0.6506
epoch 060 | loss 0.64313 | mean val AUC 0.6509
epoch 070 | loss 0.64298 | mean val AUC 0.6511
epoch 080 | loss 0.64258 | mean val AUC 0.6511
epoch 090 | loss 0.64236 | mean val AUC 0.6512
epoch 100 | loss 0.64242 | mean val AUC 0.6512
epoch 110 | loss 0.64207 | mean val AUC 0.6513
epoch 120 | loss 0.64203 | mean val AUC 0.6512
epoch 130 | loss 0.64203 | mean val AUC 0.6511
epoch 140 | loss 0.64191 | mean val AUC 0.6514
epoch 150 | loss 0.64189 | mean val AUC 0.6513


In [ ]:
display(mlp_probe_recovery_eval)

,hidden_idx,mlp_auc,mlp_val_auc_at_selected_epoch,mlp_accuracy,mlp_f1,w_hid,task_relevant,abs_w_hid,rank_abs_w
2,2,0.665045,0.645194,0.6153,0.602624,-0.648612,True,0.648612,1
0,0,0.734433,0.743609,0.6724,0.665168,0.000000,False,0.000000,2
1,1,0.601002,0.606226,0.5763,0.576427,0.000000,False,0.000000,2
3,3,0.714657,0.703468,0.6591,0.660560,0.000000,False,0.000000,2
4,4,0.565546,0.558761,0.5482,0.505906,0.000000,False,0.000000,2


## Strict Top-k Residual Probe Recovery

This supervised diagnostic asks whether each hidden concept is recoverable from only the top-k residual dimensions selected by a dense probe. It separates axis-like, sparse-mixed, and broadly distributed residual encodings.


In [ ]:
# -----------------------------
# Strict top-k residual probe recovery
# -----------------------------
# Dense probes show whether hidden concept information is linearly present in the
# residual channel. This section asks a stricter question: how many residual
# dimensions are needed to recover each hidden concept?
#
# Protocol for each hidden concept h:
# 1. Fit a dense logistic probe h ~ res_mu on train.
# 2. Rank residual dimensions by absolute dense-probe coefficient.
# 3. Refit probes using only the top-k ranked dimensions.
# 4. Evaluate every fixed top-k subset on test.
#
# Hidden labels are used here, so this is a recoverability diagnostic rather
# than an unsupervised discovery method.

TOPK_PROBE_KS = [1, 2, 3, 5, 10, 20, 50]


def fit_topk_probe_for_hidden(X_train, y_train, X_test, y_test, ks=TOPK_PROBE_KS):
    """Rank residual dimensions with a dense probe, then test strict top-k refits.
    Only fit with the top-k dimensions with highest absolute coefficient in the dense probe.
    """
    n_dims = X_train.shape[1]
    ks = sorted({min(k, n_dims) for k in ks})

    dense_clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
    )
    dense_clf.fit(X_train, y_train)
    dense_prob = dense_clf.predict_proba(X_test)[:, 1]
    dense_auc = roc_auc_score(y_test, dense_prob)

    coef = dense_clf.named_steps["logisticregression"].coef_[0]
    
    ranked_dims = np.argsort(-np.abs(coef)).astype(int).tolist()

    rows = []
    for k in ks:
        # Select the top-k dimensions based on the dense probe's absolute coefficient ranking.

        selected_dims = ranked_dims[:k]
        topk_clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=5000, class_weight="balanced", random_state=0),
        )
        topk_clf.fit(X_train[:, selected_dims], y_train)

        prob = topk_clf.predict_proba(X_test[:, selected_dims])[:, 1]
        pred = (prob >= 0.5).astype(int)
        rows.append({
            "k": k,
            "topk_auc": roc_auc_score(y_test, prob),
            "topk_accuracy": accuracy_score(y_test, pred),
            "topk_f1": f1_score(y_test, pred, zero_division=0),
            "dense_auc_from_same_probe": dense_auc,
            "topk_gap_to_dense": dense_auc - roc_auc_score(y_test, prob),
            "selected_dims": selected_dims,
        })

    return rows, ranked_dims, coef


def strict_topk_probe_train_test(train_scores, train_hidden, test_scores, test_hidden, ks=TOPK_PROBE_KS):
    X_train = to_numpy(train_scores).astype(float)
    X_test = to_numpy(test_scores).astype(float)
    H_train = to_numpy(train_hidden).astype(int)
    H_test = to_numpy(test_hidden).astype(int)

    rows = []
    coef_rows = []
    for h in range(H_train.shape[1]):
        if len(np.unique(H_train[:, h])) < 2 or len(np.unique(H_test[:, h])) < 2:
            continue

        topk_rows, ranked_dims, coef = fit_topk_probe_for_hidden(
            X_train,
            H_train[:, h],
            X_test,
            H_test[:, h],
            ks=ks,
        )
        for row in topk_rows:
            rows.append({"hidden_idx": h, **row})

        coef_rows.append({
            "hidden_idx": h,
            "ranked_dims_by_dense_coef": ranked_dims,
            "top10_dense_coef_dims": ranked_dims[:10],
            "dense_coef_l1_norm": float(np.abs(coef).sum()),
            "dense_coef_l2_norm": float(np.sqrt((coef ** 2).sum())),
        })

    return pd.DataFrame(rows), pd.DataFrame(coef_rows)


topk_probe_long, topk_probe_coef_info = strict_topk_probe_train_test(
    splits["train"]["res_mu"],
    splits["train"]["hidden_residuals"],
    splits["test"]["res_mu"],
    splits["test"]["hidden_residuals"],
)

topk_probe_long = attach_relevance(topk_probe_long, relevance)

# Wide table: one row per hidden concept, one AUC column per k.
topk_auc_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_auc")
    .rename(columns=lambda k: f"top{k}_auc")
    .reset_index()
)

topk_gap_wide = (
    topk_probe_long
    .pivot(index="hidden_idx", columns="k", values="topk_gap_to_dense")
    .rename(columns=lambda k: f"top{k}_gap_to_dense")
    .reset_index()
)

topk_probe_recovery_eval = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(topk_auc_wide, on="hidden_idx", how="left")
    .merge(topk_gap_wide, on="hidden_idx", how="left")
    .merge(topk_probe_coef_info, on="hidden_idx", how="left")
    .sort_values("rank_abs_w")
)

candidate_display_cols = [
    "hidden_idx",
    "abs_w_hid",
    "rank_abs_w",
    "axis_auc",
    "top1_auc",
    "top2_auc",
    "top3_auc",
    "top5_auc",
    "top10_auc",
    "top20_auc",
    "top50_auc",
    "distributed_auc",
    "top10_dense_coef_dims",
]
display_cols = [col for col in candidate_display_cols if col in topk_probe_recovery_eval.columns]

print("Strict top-k residual probe recovery ranked by abs(w_hid)")
display(topk_probe_recovery_eval[display_cols])

# Summarize top-k hidden concepts by true task weight, for each strict residual subset size.
summary_rows = []
for hidden_k in [1, 3, 5, 10]:
    top_hidden = topk_probe_recovery_eval.head(hidden_k)
    row = {"top_hidden_by_abs_w": hidden_k}
    row["raw_axis"] = top_hidden["axis_auc"].mean()
    for residual_k in TOPK_PROBE_KS:
        col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
        row[f"top{residual_k}_residual_dims"] = top_hidden[col].mean()
    row["dense_probe"] = top_hidden["distributed_auc"].mean()
    summary_rows.append(row)

topk_probe_summary = pd.DataFrame(summary_rows)
print("Mean hidden recovery as residual subset size increases")
display(topk_probe_summary)

# Estimate how many residual dimensions are needed to get close to the dense probe.
# A hidden concept is considered close once top-k AUC is within 0.01/0.03/0.05 of dense.
close_rows = []
for _, row in topk_probe_recovery_eval.iterrows():
    out = {
        "hidden_idx": int(row["hidden_idx"]),
        "abs_w_hid": row["abs_w_hid"],
        "rank_abs_w": int(row["rank_abs_w"]),
        "axis_auc": row["axis_auc"],
        "dense_probe_auc": row["distributed_auc"],
    }
    for tol in [0.01, 0.03, 0.05]:
        needed = np.nan
        for residual_k in TOPK_PROBE_KS:
            col = f"top{min(residual_k, to_numpy(splits['train']['res_mu']).shape[1])}_auc"
            if row[col] >= row["distributed_auc"] - tol:
                needed = residual_k
                break
        out[f"dims_needed_within_{tol:.2f}_auc"] = needed
    close_rows.append(out)

topk_compactness = pd.DataFrame(close_rows).sort_values("rank_abs_w")
print("Residual dimensions needed to approach dense-probe AUC")
display(topk_compactness)


Strict top-k residual probe recovery ranked by abs(w_hid)


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,top1_auc,top2_auc,top3_auc,top5_auc,top10_auc,top20_auc,distributed_auc,top10_dense_coef_dims
0,18,0.705692,1,0.700518,0.699438,0.701025,0.703920,0.715284,0.723139,0.728339,0.728339,"[19, 16, 0, 13, 6, 1, 2, 8, 10, 7]"
1,2,0.648612,2,0.731922,0.732040,0.737297,0.737046,0.743502,0.753280,0.754636,0.754636,"[16, 1, 19, 14, 18, 6, 13, 2, 10, 7]"
2,9,0.230210,3,0.671554,0.665360,0.668926,0.669404,0.679530,0.691581,0.695460,0.695460,"[15, 17, 13, 11, 0, 4, 3, 10, 18, 16]"
3,19,0.150507,4,0.544257,0.536699,0.564048,0.566974,0.573822,0.584057,0.586331,0.586331,"[19, 16, 10, 1, 18, 12, 3, 7, 6, 5]"
4,7,0.075182,5,0.601763,0.601763,0.609977,0.614908,0.625989,0.630301,0.639585,0.639585,"[13, 0, 8, 17, 11, 5, 6, 10, 19, 18]"
17,17,0.000000,6,0.574521,0.570079,0.581916,0.585074,0.596556,0.615036,0.618733,0.618733,"[13, 1, 16, 2, 17, 4, 18, 8, 3, 5]"
16,16,0.000000,6,0.576233,0.576233,0.576702,0.585585,0.589176,0.598024,0.610046,0.610046,"[1, 17, 4, 7, 16, 13, 12, 19, 3, 15]"
15,15,0.000000,6,0.522905,0.507289,0.573159,0.588060,0.615573,0.633723,0.641363,0.641363,"[0, 19, 2, 7, 14, 1, 8, 13, 16, 6]"
14,14,0.000000,6,0.533031,0.523818,0.540145,0.539789,0.562827,0.587434,0.596366,0.596366,"[16, 15, 19, 17, 6, 11, 4, 8, 13, 5]"
13,13,0.000000,6,0.627357,0.612771,0.612860,0.623479,0.651005,0.666031,0.668619,0.668619,"[18, 0, 8, 4, 14, 7, 15, 19, 2, 13]"


Mean hidden recovery as residual subset size increases


,top_hidden_by_abs_w,raw_axis,top1_residual_dims,top2_residual_dims,top3_residual_dims,top5_residual_dims,top10_residual_dims,top20_residual_dims,top50_residual_dims,dense_probe
0,1,0.700518,0.699438,0.701025,0.703920,0.715284,0.723139,0.728339,0.728339,0.728339
1,3,0.701332,0.698946,0.702416,0.703456,0.712772,0.722666,0.726145,0.726145,0.726145
2,5,0.650003,0.647060,0.656255,0.658450,0.667625,0.676471,0.680870,0.680870,0.680870
3,10,0.608406,0.602549,0.616605,0.621424,0.635326,0.648261,0.653948,0.653948,0.653948


Residual dimensions needed to approach dense-probe AUC


,hidden_idx,abs_w_hid,rank_abs_w,axis_auc,dense_probe_auc,dims_needed_within_0.01_auc,dims_needed_within_0.03_auc,dims_needed_within_0.05_auc
0,18,0.705692,1,0.700518,0.728339,10,1,1
1,2,0.648612,2,0.731922,0.754636,10,1,1
2,9,0.230210,3,0.671554,0.695460,10,2,1
3,19,0.150507,4,0.544257,0.586331,10,2,1
4,7,0.075182,5,0.601763,0.639585,10,2,1
17,4,0.000000,6,0.628176,0.660644,10,2,1
16,5,0.000000,6,0.647120,0.663552,10,1,1
15,6,0.000000,6,0.625382,0.668631,10,10,2
14,8,0.000000,6,0.540746,0.634099,10,5,3
13,3,0.000000,6,0.519445,0.571191,10,5,2


## Concept-Conditioned Task-Guided Residual Direction Discovery

This variant selects residual directions for their incremental task value after the task head already sees the observed concepts. The sparse head is trained on `[concepts, residual_direction_scores]`, but residual direction ranking uses only the residual coefficients.


In [ ]:
# -----------------------------
# Concept-conditioned task-guided residual direction discovery
# -----------------------------
# The previous task-guided section selected residual directions using y alone.
# That can favor broad global task directions. Here we control for the known
# concept bottleneck: the sparse task head sees both observed concepts and
# candidate residual directions, and we rank only the residual-direction weights.
#
# Hidden labels are still used only after selection, for validation matching and
# held-out test evaluation.

CONDITIONED_TASK_HEAD_C_GRID = TASK_HEAD_C_GRID if "TASK_HEAD_C_GRID" in globals() else [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
CONDITIONED_TASK_TOP_MS = TASK_DISCOVERY_TOP_MS if "TASK_DISCOVERY_TOP_MS" in globals() else [5, 10, 20, 50]
CONDITIONED_TASK_HEAD_AUC_TOL = TASK_HEAD_AUC_TOL if "TASK_HEAD_AUC_TOL" in globals() else 0.005


def get_concept_conditioning_source(splits):
    """Choose the concept representation used as conditioning input.

    Prefer model-predicted concept means (`c_mu`) when the joint concept/residual
    channel was saved. Fall back to predicted probabilities/sample means if present,
    and only use ground-truth dataset concepts as an oracle fallback.
    """
    preferred_keys = ["c_mu", "concept_probs", "concept_sample_mean", "concepts"]
    for key in preferred_keys:
        if all(key in splits[split] for split in ["train", "val", "test"]):
            return key
    raise KeyError(f"No concept conditioning source found. Tried: {preferred_keys}")


def standardize_concepts_for_conditioning(train_concepts, val_concepts, test_concepts):
    """Standardize concept inputs using train statistics only."""
    C_train = to_numpy(train_concepts).astype(float)
    C_val = to_numpy(val_concepts).astype(float)
    C_test = to_numpy(test_concepts).astype(float)
    mean = C_train.mean(axis=0, keepdims=True)
    std = C_train.std(axis=0, keepdims=True) + 1e-8
    return (C_train - mean) / std, (C_val - mean) / std, (C_test - mean) / std


def fit_concept_conditioned_sparse_task_head(
    C_train,
    Z_train,
    y_train,
    C_val,
    Z_val,
    y_val,
    c_grid=CONDITIONED_TASK_HEAD_C_GRID,
):
    """
    Train sparse task heads on [concepts, residual-direction scores].

    Model selection uses validation y-AUC. Residual direction ranking uses only
    coefficients attached to Z, so observed-concept coefficients do not directly
    become discovered residual concepts.
    """
    y_train = to_numpy(y_train).astype(int).reshape(-1)
    y_val = to_numpy(y_val).astype(int).reshape(-1)
    X_train = np.concatenate([C_train, Z_train], axis=1)
    X_val = np.concatenate([C_val, Z_val], axis=1)
    n_concepts = C_train.shape[1]

    rows = []
    for C in c_grid:
        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                solver="saga",
                l1_ratio=1.0,
                C=C,
                class_weight="balanced",
                max_iter=5000,
                tol=1e-3,
                random_state=0,
            ),
        )
        clf.fit(X_train, y_train)
        val_prob = clf.predict_proba(X_val)[:, 1]
        val_auc = roc_auc_score(y_val, val_prob)

        coef = clf.named_steps["logisticregression"].coef_[0]
        concept_coef = coef[:n_concepts]
        residual_coef = coef[n_concepts:]
        rows.append({
            "C": C,
            "val_task_auc": val_auc,
            "n_nonzero_total": int((np.abs(coef) > 1e-8).sum()),
            "n_nonzero_concepts": int((np.abs(concept_coef) > 1e-8).sum()),
            "n_nonzero_residual_directions": int((np.abs(residual_coef) > 1e-8).sum()),
            "clf": clf,
            "coef": coef,
            "residual_coef": residual_coef,
        })

    best_auc = max(row["val_task_auc"] for row in rows)
    eligible = [row for row in rows if row["val_task_auc"] >= best_auc - CONDITIONED_TASK_HEAD_AUC_TOL]
    best = min(eligible, key=lambda row: (row["n_nonzero_residual_directions"], row["n_nonzero_total"], -row["val_task_auc"], row["C"]))

    residual_coef = best["residual_coef"]
    ranked_direction_ids = np.argsort(-np.abs(residual_coef)).astype(int).tolist()
    nonzero_ranked_direction_ids = [idx for idx in ranked_direction_ids if abs(residual_coef[idx]) > 1e-8]
    best["nonzero_ranked_direction_ids"] = nonzero_ranked_direction_ids
    tuning = pd.DataFrame([{k: v for k, v in row.items() if k not in ["clf", "coef", "residual_coef"]} for row in rows])
    return best, tuning, ranked_direction_ids


# Reuse the candidate directions from the task-guided discovery section when available;
# otherwise build them here from train residuals.
if "candidate_direction_sets" not in globals():
    X_train_disc, X_val_disc, X_test_disc, disc_mean, disc_std = standardize_residual_splits_for_discovery(
        splits["train"]["res_mu"],
        splits["val"]["res_mu"],
        splits["test"]["res_mu"],
    )
    candidate_direction_sets = build_residual_direction_candidates(X_train_disc)



concept_conditioning_key = get_concept_conditioning_source(splits)
print(f"Conditioning task head on concept source: {concept_conditioning_key}")

C_train_cond, C_val_cond, C_test_cond = standardize_concepts_for_conditioning(
    splits["train"][concept_conditioning_key],
    splits["val"][concept_conditioning_key],
    splits["test"][concept_conditioning_key],
)

conditioned_task_guided_details = []
conditioned_task_guided_summaries = []
conditioned_task_head_tuning_tables = {}

for generator_name, directions in candidate_direction_sets.items():
    print("=" * 80)
    print(f"Concept-conditioned generator: {generator_name} | residual directions: {directions.shape[0]}")

    Z_train = project_residual_directions(X_train_disc, directions)
    Z_val = project_residual_directions(X_val_disc, directions)
    Z_test = project_residual_directions(X_test_disc, directions)

    task_head, tuning_table, ranked_direction_ids = fit_concept_conditioned_sparse_task_head(
        C_train_cond,
        Z_train,
        splits["train"]["y"],
        C_val_cond,
        Z_val,
        splits["val"]["y"],
    )
    conditioned_task_head_tuning_tables[generator_name] = tuning_table

    X_test_task = np.concatenate([C_test_cond, Z_test], axis=1)
    test_task_prob = task_head["clf"].predict_proba(X_test_task)[:, 1]
    test_task_auc = roc_auc_score(to_numpy(splits["test"]["y"]).astype(int), test_task_prob)
    print(
        f"selected C={task_head['C']} | val y-AUC={task_head['val_task_auc']:.3f} | "
        f"test y-AUC={test_task_auc:.3f} | nonzero concepts={task_head['n_nonzero_concepts']} | "
        f"nonzero residual dirs={task_head['n_nonzero_residual_directions']}"
    )

    selectable_direction_ids = task_head["nonzero_ranked_direction_ids"]
    if len(selectable_direction_ids) == 0:
        # If the sparse task head assigns no residual weight, there is no
        # concept-conditioned residual discovery to evaluate for this generator.
        print("No nonzero residual directions selected by the conditioned task head; skipping hidden recovery.")
        continue

    # Evaluate each effective top_m once. This avoids duplicate rows when
    # top_m requests exceed the number of nonzero residual directions selected
    # by the sparse conditioned task head.
    effective_top_ms = sorted({min(m, len(selectable_direction_ids)) for m in CONDITIONED_TASK_TOP_MS})
    for top_m in effective_top_ms:
        selected_direction_ids = selectable_direction_ids[:top_m]

        val_matches = match_selected_directions_on_val(
            Z_val,
            splits["val"]["hidden_residuals"],
            selected_direction_ids,
        )
        test_eval = evaluate_fixed_direction_matches_on_test(
            Z_test,
            splits["test"]["hidden_residuals"],
            val_matches,
        )
        test_eval = attach_relevance(test_eval, relevance).sort_values("rank_abs_w")
        test_eval["generator"] = generator_name
        test_eval["top_m_task_selected_directions"] = top_m
        test_eval["conditioned_task_head_val_auc"] = task_head["val_task_auc"]
        test_eval["conditioned_task_head_test_auc"] = test_task_auc
        test_eval["concept_conditioning_source"] = concept_conditioning_key
        test_eval["conditioned_nonzero_concepts"] = task_head["n_nonzero_concepts"]
        test_eval["conditioned_nonzero_residual_directions"] = task_head["n_nonzero_residual_directions"]
        test_eval["selected_direction_ids"] = [selected_direction_ids] * len(test_eval)
        conditioned_task_guided_details.append(test_eval)

        top1 = test_eval.head(1)
        top3 = test_eval.head(3)
        top5 = test_eval.head(5)
        conditioned_task_guided_summaries.append({
            "generator": generator_name,
            "top_m_task_selected_directions": top_m,
            "conditioned_task_head_val_auc": task_head["val_task_auc"],
            "conditioned_task_head_test_auc": test_task_auc,
            "concept_conditioning_source": concept_conditioning_key,
            "conditioned_nonzero_concepts": task_head["n_nonzero_concepts"],
            "conditioned_nonzero_residual_directions": task_head["n_nonzero_residual_directions"],
            "top1_mean_discovery_auc": top1["discovery_auc"].mean(),
            "top3_mean_discovery_auc": top3["discovery_auc"].mean(),
            "top5_mean_discovery_auc": top5["discovery_auc"].mean(),
            "top5_unique_matched_directions": int(test_eval.head(5)["discovered_direction_idx"].nunique()),
            "top5_matched_direction_ids": test_eval.head(5)["discovered_direction_idx"].tolist(),
        })

conditioned_task_guided_recovery_eval = pd.concat(conditioned_task_guided_details, ignore_index=True)
conditioned_task_guided_summary = pd.DataFrame(conditioned_task_guided_summaries).sort_values(
    ["top5_mean_discovery_auc", "top3_mean_discovery_auc", "top5_unique_matched_directions"],
    ascending=[False, False, False],
)

print("Concept-conditioned task-guided residual direction discovery summary")
display(conditioned_task_guided_summary)

best_conditioned_task_guided = conditioned_task_guided_summary.iloc[0]
best_conditioned_task_guided_detail = conditioned_task_guided_recovery_eval[
    (conditioned_task_guided_recovery_eval["generator"] == best_conditioned_task_guided["generator"])
    & (
        conditioned_task_guided_recovery_eval["top_m_task_selected_directions"]
        == best_conditioned_task_guided["top_m_task_selected_directions"]
    )
].sort_values("rank_abs_w")

print("Best concept-conditioned task-guided discovery detail ranked by abs(w_hid)")
display(best_conditioned_task_guided_detail[[
    "generator",
    "top_m_task_selected_directions",
    "hidden_idx",
    "discovered_direction_idx",
    "discovery_auc",
    "val_discovery_auc",
    "w_hid",
    "abs_w_hid",
    "rank_abs_w",
    "conditioned_task_head_test_auc",
    "concept_conditioning_source",
    "conditioned_nonzero_concepts",
    "conditioned_nonzero_residual_directions",
]])

conditioned_task_guided_comparison = (
    recovery_eval[["hidden_idx", "axis_auc", "abs_w_hid", "rank_abs_w"]]
    .merge(probe_recovery_eval[["hidden_idx", "distributed_auc"]], on="hidden_idx", how="left")
    .merge(
        best_conditioned_task_guided_detail[["hidden_idx", "discovery_auc", "discovered_direction_idx"]],
        on="hidden_idx",
        how="left",
    )
    .sort_values("rank_abs_w")
)

print("Raw axis vs dense probe vs best concept-conditioned task-guided discovery")
display(conditioned_task_guided_comparison)

best_conditioned_task_guided_summary = pd.DataFrame({
    "method": ["raw_axis", "dense_probe", "concept_conditioned_task_guided"],
    "top1_mean_auc": [
        conditioned_task_guided_comparison.head(1)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(1)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(1)["discovery_auc"].mean(),
    ],
    "top3_mean_auc": [
        conditioned_task_guided_comparison.head(3)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(3)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(3)["discovery_auc"].mean(),
    ],
    "top5_mean_auc": [
        conditioned_task_guided_comparison.head(5)["axis_auc"].mean(),
        conditioned_task_guided_comparison.head(5)["distributed_auc"].mean(),
        conditioned_task_guided_comparison.head(5)["discovery_auc"].mean(),
    ],
})
display(best_conditioned_task_guided_summary)


Conditioning task head on concept source: c_mu
Concept-conditioned generator: raw_axes | residual directions: 20
selected C=0.01 | val y-AUC=0.894 | test y-AUC=0.895 | nonzero concepts=4 | nonzero residual dirs=13
Concept-conditioned generator: pca | residual directions: 20
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=0 | nonzero residual dirs=1
Concept-conditioned generator: sparse_pca | residual directions: 20
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=0 | nonzero residual dirs=9
Concept-conditioned generator: combined_raw_pca_sparsepca | residual directions: 60
selected C=0.001 | val y-AUC=0.894 | test y-AUC=0.894 | nonzero concepts=0 | nonzero residual dirs=18
Concept-conditioned task-guided residual direction discovery summary


,generator,top_m_task_selected_directions,conditioned_task_head_val_auc,conditioned_task_head_test_auc,concept_conditioning_source,conditioned_nonzero_concepts,conditioned_nonzero_residual_directions,top1_mean_discovery_auc,top3_mean_discovery_auc,top5_mean_discovery_auc,top5_unique_matched_directions,top5_matched_direction_ids
5,sparse_pca,9,0.894253,0.894410,c_mu,0,9,0.700518,0.701418,0.650055,4,"[17, 18, 11, 14, 17]"
7,combined_raw_pca_sparsepca,10,0.894268,0.894282,c_mu,0,18,0.700518,0.701418,0.650055,4,"[57, 58, 11, 10, 57]"
8,combined_raw_pca_sparsepca,18,0.894268,0.894282,c_mu,0,18,0.700518,0.701418,0.650055,4,"[57, 58, 11, 10, 57]"
1,raw_axes,10,0.894318,0.894614,c_mu,4,13,0.700518,0.701332,0.650003,4,"[13, 6, 11, 10, 13]"
2,raw_axes,13,0.894318,0.894614,c_mu,4,13,0.700518,0.701332,0.650003,4,"[13, 6, 11, 10, 13]"
4,sparse_pca,5,0.894253,0.894410,c_mu,0,9,0.697597,0.700444,0.648909,4,"[16, 18, 11, 14, 11]"
0,raw_axes,5,0.894318,0.894614,c_mu,4,13,0.697597,0.700397,0.648881,4,"[9, 16, 11, 10, 11]"
6,combined_raw_pca_sparsepca,5,0.894268,0.894282,c_mu,0,18,0.695424,0.697791,0.646740,4,"[40, 58, 40, 10, 20]"
3,pca,1,0.893956,0.893600,c_mu,0,1,0.694837,0.696239,0.645377,1,"[0, 0, 0, 0, 0]"


Best concept-conditioned task-guided discovery detail ranked by abs(w_hid)


,generator,top_m_task_selected_directions,hidden_idx,discovered_direction_idx,discovery_auc,val_discovery_auc,w_hid,abs_w_hid,rank_abs_w,conditioned_task_head_test_auc,concept_conditioning_source,conditioned_nonzero_concepts,conditioned_nonzero_residual_directions
100,sparse_pca,9,18,17,0.700518,0.694672,-0.705692,0.705692,1,0.89441,c_mu,0,9
101,sparse_pca,9,2,18,0.732182,0.740729,-0.648612,0.648612,2,0.89441,c_mu,0,9
102,sparse_pca,9,9,11,0.671554,0.651557,0.230210,0.230210,3,0.89441,c_mu,0,9
103,sparse_pca,9,19,14,0.544257,0.546242,-0.150507,0.150507,4,0.89441,c_mu,0,9
104,sparse_pca,9,7,17,0.601763,0.606747,0.075182,0.075182,5,0.89441,c_mu,0,9
117,sparse_pca,9,17,17,0.570079,0.560857,0.000000,0.000000,6,0.89441,c_mu,0,9
116,sparse_pca,9,16,16,0.573661,0.560417,0.000000,0.000000,6,0.89441,c_mu,0,9
115,sparse_pca,9,15,9,0.522905,0.519063,0.000000,0.000000,6,0.89441,c_mu,0,9
114,sparse_pca,9,14,17,0.532370,0.520966,0.000000,0.000000,6,0.89441,c_mu,0,9
113,sparse_pca,9,13,17,0.627357,0.618274,0.000000,0.000000,6,0.89441,c_mu,0,9


Raw axis vs dense probe vs best concept-conditioned task-guided discovery


,hidden_idx,axis_auc,abs_w_hid,rank_abs_w,distributed_auc,discovery_auc,discovered_direction_idx
0,18,0.700518,0.705692,1,0.728339,0.700518,17
1,2,0.731922,0.648612,2,0.754636,0.732182,18
2,9,0.671554,0.230210,3,0.695460,0.671554,11
3,19,0.544257,0.150507,4,0.586331,0.544257,14
4,7,0.601763,0.075182,5,0.639585,0.601763,17
17,17,0.574521,0.000000,6,0.618733,0.570079,17
16,16,0.576233,0.000000,6,0.610046,0.573661,16
15,15,0.522905,0.000000,6,0.641363,0.522905,9
14,14,0.533031,0.000000,6,0.596366,0.532370,17
13,13,0.627357,0.000000,6,0.668619,0.627357,17


,method,top1_mean_auc,top3_mean_auc,top5_mean_auc
0,raw_axis,0.700518,0.701332,0.650003
1,dense_probe,0.728339,0.726145,0.680870
2,concept_conditioned_task_guided,0.700518,0.701418,0.650055
